# 07 — Motor de corrida del Experimento B y corrida de humo con C1

**Autor:** Jason Sepúlveda S.
**Proyecto:** Evaluación experimental de la extracción automática y la representación
persistente de decisiones organizacionales frente a Recuperación Aumentada (RAG)
**Depende de:** [`06-jss-ajuste-c1.ipynb`](06-jss-ajuste-c1.ipynb) (C1 ajustada) ·
[`05-jss-seleccion-juez.ipynb`](05-jss-seleccion-juez.ipynb) (modelo juez) ·
[ADR 0005](../docs/decisions/0005-adr-development-evaluation-split.md)

---

> **Estado: corrida de humo de desarrollo.** Se ejecutó el motor una vez sobre las 21 preguntas
> de calibración de ES2015 marcadas para uso, con la condición C1. ES2015 es una **serie de
> desarrollo** (ADR 0005): **estas cifras no entran en ninguna tabla de resultados** ni
> estiman el desempeño de C1. Sirven para comprobar que el motor funciona de punta a punta y
> para detectar problemas de medición **antes** del experimento real.

# 1. Un caso concreto: respuesta correcta, atribución perdida y citación correcta pero poco precisa

La pregunta q002 de ES2015 pide el precio de venta fijado para el control remoto. C1 responde
«twenty five Euros» (veinticinco euros), y el juez automático la da por **correcta**. La métrica
de **citación** (qué fragmentos de la reunión respaldan la respuesta) la cuenta como
**correcta**, pero con una **precisión baja**: la mayoría de lo citado no contiene la evidencia
de referencia. La celda siguiente lo muestra con los datos de la corrida.

Tres términos que se usan en todo el notebook:

- **Citación**: la lista de fragmentos de la transcripción que se presentan como respaldo de
  la respuesta. Se evalúa de forma determinista, sin modelo, según el
  [ADR 0009](../docs/decisions/0009-adr-regla-de-citacion-correcta.md): es **correcta si al
  menos uno** de los pasajes citados comparte actos de diálogo con la evidencia de referencia de
  la pregunta. La **precisión de la citación** es la fracción de los pasajes citados que lo
  hacen, y se informa aparte como cifra secundaria.
- **Atribución**: el modo en que C1 sabe qué fragmentos usó el modelo. Se le pide que termine
  su respuesta con una línea `USED: 2, 3` (los números de los fragmentos que utilizó). Si el
  sistema lee esa línea, la atribución es `reported` (informada); si no la encuentra, es
  `absent` (ausente) y, por defecto, se citan **todos** los fragmentos recuperados.
- **Abstención**: C1 responde que el contexto no contiene la respuesta
  (`no_answer_in_context`) en lugar de intentar contestar.

In [1]:
import json
from dataclasses import asdict

import pandas as pd

from afg.experiments.citation_analysis import rescore_citation
from afg.experiments.questions import load_experiment_questions
from afg.experiments.runner import MANIFEST_FILE, RECORDS_FILE, load_records
from afg.experiments.summary import (
    citation_tally,
    inline_marker_rows,
    record_rows,
    resume_status,
    stratum_summary,
    wiring_summary,
)
from afg.shared.paths import EXPERIMENT_B_DIR, PROJECT_ROOT

pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 80)

RUN_DIR = EXPERIMENT_B_DIR / "humo-c1-es2015"
QUESTIONS_CSV = PROJECT_ROOT / "data" / "processed" / "judge-calibration" / "ES2015.preguntas.csv"
for needed in (RUN_DIR / RECORDS_FILE, QUESTIONS_CSV):
    if not needed.exists():
        raise SystemExit(
            f"No se encontro {needed}.\n"
            "Se genera con `uv run afg experiment b --help` (requiere Ollama) y con la hoja "
            "de calibracion de ES2015."
        )

MANIFEST = json.loads((RUN_DIR / MANIFEST_FILE).read_text(encoding="utf-8"))
ALL_LINES = load_records(RUN_DIR / RECORDS_FILE)
QUESTIONS = {q.id: q for q in load_experiment_questions(QUESTIONS_CSV)}
# Los registros de la corrida de humo se escribieron con la regla anterior (todas las citas
# debian tocar la referencia). La citacion se recalcula con la regla vigente (ADR 0009) a partir
# de las respuestas guardadas y de las preguntas; el archivo de la corrida no se modifica.
RECORDS = rescore_citation(load_records(RUN_DIR / RECORDS_FILE, latest_only=True), QUESTIONS)
ROWS = pd.DataFrame([asdict(r) for r in record_rows(RECORDS)])
print(f"Corrida en {RUN_DIR.relative_to(PROJECT_ROOT)}: {len(RECORDS)} registros")

Corrida en reports/experiment-b/humo-c1-es2015: 21 registros


In [2]:
q002 = next(r for r in RECORDS if r["question_id"].endswith(".q002"))
print("Pregunta   :", q002["trace"]["question"])
print("Respuesta  :", q002["answer"]["text"])
print(
    "Juez       : correcta =",
    q002["verdict"]["correct"],
    "| respaldada =",
    q002["verdict"]["evidence_supported"],
)
print("Atribucion :", q002["trace"]["attribution"])
print("Fragmentos citados:", len(q002["trace"]["cited_chunk_ids"]))
print("Citacion correcta  :", q002["citation_correct"])
print("Precision de la citacion:", round(q002["citation_precision"], 2))

Pregunta   : What selling price was set for the remote control?
Respuesta  : The selling price that was set for the remote control is twenty five Euros. USED: 2, 3
Juez       : correcta = True | respaldada = True
Atribucion : absent
Fragmentos citados: 5
Citacion correcta  : True
Precision de la citacion: 0.33


La respuesta es correcta y la citación también, pero la atribución quedó `absent`: por eso se
citaron todos los fragmentos recuperados y no solo los dos que el modelo declaró haber usado. La
citación pasa porque, citando todo, alguno contiene la referencia; la **precisión** muestra que
la mayor parte de lo citado es ajeno. Por eso las dos cifras se leen juntas (sección 6). La
sección 5 explica por qué la atribución se perdió.

---

# 2. Qué se ejecutó y cómo está cableado

El motor combina preguntas × condiciones × corridas. Para cada combinación obtiene la
respuesta, la juzga con un modelo distinto del generador, calcula la citación y agrega una
línea al archivo `respuestas.jsonl`. El `manifest.json` guarda la configuración, la partición
(desarrollo o evaluación) y los conteos. La celda muestra la configuración leída del manifiesto
y, aparte, los valores distintos que aparecen en los registros: si hay un solo valor por
ajuste, el ajuste fue constante.

In [3]:
pd.DataFrame(
    [
        ("Corrida", MANIFEST["run_name"]),
        ("Serie(s)", ", ".join(MANIFEST["series"])),
        ("Partición", MANIFEST["partition"]),
        ("Solo desarrollo", MANIFEST["development_only"]),
        ("Nota del manifiesto", MANIFEST["note"]),
        ("Commit del código", MANIFEST["git_commit"][:7]),
        *[(f"settings.{k}", v) for k, v in MANIFEST["settings"].items()],
    ],
    columns=["campo", "valor"],
)

,campo,valor
0,Corrida,humo-c1-es2015
1,Serie(s),ES2015
2,Partición,development
3,Solo desarrollo,True
4,Nota del manifiesto,Development series (ADR 0005): these figures never enter a results table.
5,Commit del código,0e1809a
6,settings.generative_model,qwen3:8b
7,settings.temperature,0.1
8,settings.base_seed,42
9,settings.runs,1


In [4]:
pd.Series(wiring_summary(RECORDS), name="valores distintos en los registros").to_frame()

,valores distintos en los registros
seeds,[42]
seed_applied,[True]
generative_models,[qwen3:8b]
temperatures,[0.1]
partitions,[development]
judge_models,[gpt-oss:20b]
judge_orders,[reference_first]
judge_seeds,[42]
judge_temperatures,[0.0]
conditions,[C1]


Lectura: un único generador (`qwen3:8b`, temperatura 0,1), un único juez distinto
(`gpt-oss:20b`, temperatura 0, orden «referencia primero») con una semilla fija, y una sola
semilla de generación aplicada (`seed_applied` verdadero). El juez es una **constante** del
experimento y no una fuente de variación entre registros.

---

# 3. Un registro por pregunta

Columnas: **estrato** (E1 hecho puntual, E2, E3 y E4 pregunta sin respuesta en la reunión);
**resultado** de C1; **atribución**; veredicto del juez (`correcta` y `respaldada`, esta última
indica si la transcripción sostiene lo afirmado); **citación correcta** (determinista; vacía
cuando no aplica, p. ej. en una abstención), junto con su **precisión**; número de fragmentos citados; y los segundos de
respuesta y de juicio.

In [5]:
TABLE = ROWS.rename(
    columns={
        "question": "pregunta",
        "question_text": "texto",
        "stratum": "estrato",
        "outcome": "resultado",
        "attribution": "atribución",
        "correct": "correcta",
        "evidence_supported": "respaldada",
        "citation_correct": "citación correcta",
        "citation_precision": "precisión de la citación",
        "cited_chunks": "fragmentos citados",
        "answer_seconds": "s respuesta",
        "judge_seconds": "s juicio",
    }
)
TABLE.round({"precisión de la citación": 2, "s respuesta": 1, "s juicio": 1})

,pregunta,texto,estrato,resultado,atribución,correcta,respaldada,citación correcta,precisión de la citación,fragmentos citados,s respuesta,s juicio
0,q002,What selling price was set for the remote control?,E1,answered,absent,True,True,True,0.33,5,20.5,14.3
1,q003,Which market was the remote control intended for?,E1,answered,reported,True,False,True,0.33,5,14.2,6.8
2,q004,What was the maximum allowed production cost per remote control?,E1,answered,reported,True,True,True,1.00,1,9.8,2.9
3,q006,Which feature did management decide to leave out because it was considered o...,E1,answered,reported,True,True,True,1.00,2,11.0,2.5
4,q007,Which company branding elements had to be incorporated into the remote control?,E1,answered,reported,True,False,True,0.50,2,11.0,3.6
5,q008,Was an LCD included in the remote control design?,E1,no_answer_in_context,absent,False,True,None,NaN,0,4.0,4.8
6,q010,What feature did the team approve to help find a misplaced remote control?,E1,answered,reported,True,True,False,0.00,2,9.1,3.8
7,q011,What lighting feature did the team decide to incorporate into the remote con...,E1,answered,reported,False,False,False,0.00,2,13.3,4.3
8,q013,What type of user interface was chosen for the remote control?,E1,answered,reported,True,False,False,0.00,2,10.9,2.9
9,q016,What energy source was chosen for the remote control?,E1,no_answer_in_context,absent,False,False,None,NaN,0,4.1,3.0


---

# 4. Resumen por estrato y tiempos

La **tasa de abstención** es la fracción de preguntas en que C1 declinó contestar. En E4
(preguntas sin respuesta en la reunión) abstenerse es lo esperable, de modo que allí la
«correcta» mide si el sistema evitó inventar una respuesta. En los demás estratos una
abstención cuenta como respuesta incorrecta.

In [6]:
STRATA = pd.DataFrame([asdict(s) for s in stratum_summary(RECORDS)])
STRATA.rename(
    columns={
        "stratum": "estrato",
        "n": "preguntas",
        "answered": "respondidas",
        "abstained": "abstenciones",
        "correct": "correctas",
        "correct_rate": "tasa correcta",
        "abstention_rate": "tasa abstención",
        "mean_answer_seconds": "s respuesta (media)",
        "mean_judge_seconds": "s juicio (media)",
        "total_seconds": "s totales",
    }
).round(2)

,estrato,preguntas,respondidas,abstenciones,correctas,tasa correcta,tasa abstención,s respuesta (media),s juicio (media),s totales
0,E1,14,11,3,10,0.71,0.21,10.37,4.78,212.04
1,E2,2,2,0,2,1.00,0.00,11.31,4.21,31.04
2,E3,1,0,1,0,0.00,1.00,3.81,3.92,7.72
3,E4,4,0,4,4,1.00,1.00,3.06,2.18,20.96


In [7]:
print(
    f"Tiempo total de la corrida: {ROWS.answer_seconds.sum() + ROWS.judge_seconds.sum():.0f} s "
    f"({ROWS.answer_seconds.sum():.0f} s respondiendo, {ROWS.judge_seconds.sum():.0f} s juzgando)"
)

Tiempo total de la corrida: 272 s (184 s respondiendo, 88 s juzgando)


Con 2 preguntas en E2 y 1 en E3, las tasas por estrato **no son estimaciones**: son conteos de
una verificación de funcionamiento. Lo que sí se aprecia es el orden de magnitud del costo: las
respuestas con contexto tardan del orden de diez segundos y las abstenciones, unos pocos.

---

# 5. El marcador `USED:` en la misma línea

OpenKOS v0.5.4 solo reconoce el marcador `USED:` cuando está **solo en su línea**. Si el modelo
lo escribe al final de la frase («… twenty five Euros. USED: 2, 3»), el sistema no lo lee, la
atribución queda `absent` y se citan todos los fragmentos recuperados. Esto **solo afecta a la
métrica de citación**: la respuesta y su veredicto de corrección no cambian. La celda detecta
esos casos en los datos.

In [8]:
INLINE = pd.DataFrame(
    [
        {
            "pregunta": r.question,
            "atribución": r.attribution,
            "correcta": r.correct,
            "fragmentos citados": r.cited_chunks,
            "citación correcta": r.citation_correct,
            "precisión": None if r.citation_precision is None else round(r.citation_precision, 2),
        }
        for r in inline_marker_rows(RECORDS)
    ]
)
INLINE

,pregunta,atribución,correcta,fragmentos citados,citación correcta,precisión
0,q002,absent,True,5,True,0.33
1,q020,absent,True,5,True,0.50


In [9]:
for r in RECORDS:
    if r["question_id"].split(".")[-1] in set(INLINE["pregunta"]):
        print(r["question_id"], "->", repr(r["trace"]["raw_reply"][-60:]))

ES2015.q002 -> ' set for the remote control is twenty five Euros. USED: 2, 3'
ES2015.q020 -> 'contrast are not necessary for the basic version. USED: 3, 5'


La corrección ya está planteada en OpenKOS
([jasonssdev/openkos#1384](https://github.com/jasonssdev/openkos/issues/1384)) y el pendiente
propio está en [jasonssdev/mcd-afg#87](https://github.com/jasonssdev/mcd-afg/issues/87). C1
**no** la adelanta por su cuenta: la incorporará cuando se publique, para que las tres
condiciones compartan la misma versión del sistema de respuesta y la comparación sea justa.

# 6. La regla de citación y su precisión

Según el [ADR 0009](../docs/decisions/0009-adr-regla-de-citacion-correcta.md),
`afg.evaluation.citation.citation_correct` da la citación por correcta cuando **al menos un**
pasaje citado comparte un acto de diálogo con la evidencia de referencia, y
`citation_precision` informa la fracción de los pasajes citados que lo hace. La regla anterior
exigía que *cada* pasaje coincidiera; el análisis que llevó al cambio está en el
[notebook 09](09-jss-regla-de-citacion.ipynb). La celda cuenta, entre las respuestas con al
menos una cita, cuántas fallan y cuántas de esas fallas corresponden a respuestas correctas o al
marcador en línea de la sección 5, y la precisión media de las citas.

In [10]:
T = citation_tally(RECORDS)
pd.Series(
    {
        "respuestas con citas evaluadas": T.evaluated,
        "citación correcta": T.correct,
        "citación incorrecta": T.incorrect,
        "tasa de citación incorrecta": round(T.incorrect_rate, 3),
        "incorrectas con marcador en línea (sección 5)": T.incorrect_with_inline_marker,
        "incorrectas aunque la respuesta es correcta": T.incorrect_with_correct_answer,
        "precisión media de las citas": round(T.mean_precision, 2),
    },
    name="valor",
    dtype=object,
).to_frame()

,valor
respuestas con citas evaluadas,13
citación correcta,9
citación incorrecta,4
tasa de citación incorrecta,0.308
incorrectas con marcador en línea (sección 5),0
incorrectas aunque la respuesta es correcta,3
precisión media de las citas,0.51


**Riesgo declarado (ADR 0009).** Citar de más no hace fallar la citación, de modo que una
condición que cita todo la pasa; la precisión es la cifra que lo deja a la vista. Además, la
granularidad difiere entre condiciones: C1 cita fragmentos completos y C2 y C3 citarán
decisiones, con evidencia más acotada, de modo que una regla basada en el solapamiento de actos
podría favorecer a C1. Se revisará con los primeros datos de C2, antes de correr cualquier
evaluación, y cualquier cambio se registrará antes de ver resultados de las series de
evaluación.

---

# 7. Reanudación

El archivo de registros solo se agrega al final; si una combinación se repite, vale la última
línea, y una combinación con error se reintenta en la siguiente ejecución. La celda compara las
líneas del archivo con los registros vigentes y con el total esperado: si no queda trabajo
pendiente, volver a ejecutar el mismo comando no escribe nada.

In [11]:
COUNTS = MANIFEST["counts"]
S = resume_status(RECORDS, expected=COUNTS["expected"])
pd.Series(
    {
        "líneas en el archivo": len(ALL_LINES),
        "registros vigentes (última línea por clave)": len(RECORDS),
        "esperados": S.expected,
        "completos": S.complete,
        "con error": S.errors,
        "pendientes": S.pending,
        "manifiesto: completos / errores": f"{COUNTS['complete']} / {COUNTS['errors']}",
    },
    name="valor",
).to_frame()

,valor
líneas en el archivo,21
registros vigentes (última línea por clave),21
esperados,21
completos,21
con error,0
pendientes,0
manifiesto: completos / errores,21 / 0


El comportamiento de reanudación (saltar lo completo, reintentar lo fallido, ignorar una última
línea truncada) está cubierto por las pruebas del motor
(`tests/test_experiment_b_runner.py`); aquí se verifica solo que el estado de esta corrida es
coherente.

---

# 8. Qué se concluye y qué no

- El motor corre de punta a punta con C1 y el juez reales: 21 preguntas, ninguna con error, y
  el estado queda registrado y es reanudable.
- **No** se concluye nada sobre la calidad de C1: ES2015 es de desarrollo y son 21 preguntas.
- La métrica de citación tiene dos asuntos abiertos antes del experimento real: el marcador
  `USED:` en línea (corrección a incorporar cuando se publique; #87) y la revisión del riesgo de
  granularidad entre condiciones con los primeros datos de C2 (sección 6, ADR 0009).
- Cada cifra de este notebook proviene de `afg.experiments.summary` y tiene prueba.